# 01 · Phase 1 — Download CirCor DigiScope v1.0.3 (and verify it)

**Dataset:** *The CirCor DigiScope Phonocardiogram Dataset*, version **1.0.3** only
(never mixed with 1.0.0 / 1.0.1 / 1.0.2) — https://physionet.org/content/circor-heart-sound/1.0.3/

Official command (PhysioNet / Challenge 2022 documentation):
```bash
wget -r -N -c -np https://physionet.org/files/circor-heart-sound/1.0.3/
```
`wget` is not installed on this Windows machine and PhysioNet throttles each connection
(~40 KB/s), so the cell below does the same thing — mirror every file listed in
`SHA256SUMS.txt` — with `curl`/`urllib`, 24 parallel connections, resume, and **SHA-256
verification of every file**. Re-running it only re-downloads missing/corrupt files.

| File | Use in project |
|---|---|
| `training_data/*.wav` | raw PCG, 4 kHz, one per auscultation location |
| `training_data/*.hea` | WFDB header: sampling rate, sample count |
| `training_data/*.tsv` | S1 / systole / S2 / diastole segmentation (label 0 = unannotated) |
| `training_data/*.txt` | subject-level: murmur, murmur locations, timing/shape/grade/pitch/quality, outcome |
| `training_data.csv` | subject metadata table |
| `RECORDS`, `SHA256SUMS.txt`, `LICENSE.txt` | record list, integrity check, license (ODC-By 1.0) |

In [1]:
%run ../common/00_config.ipynb
use_half("midsem", oracle="classical")

Python environment is working
PyTorch: 2.14.0+cu126


CUDA available: True
GPU: NVIDIA GeForce RTX 4050 Laptop GPU


CirCor version 1.0.3
midsem: results -> results/midsem/ | reward oracle: classical


CLI check that PhysioNet is reachable and the release is 1.0.3:

In [2]:
!curl -sI https://physionet.org/files/circor-heart-sound/1.0.3/SHA256SUMS.txt | findstr /i "HTTP content-length"

HTTP/1.1 200 OK
Content-Length: 957290


In [3]:
import hashlib
import urllib.request
from concurrent.futures import ThreadPoolExecutor, as_completed

BASE_URL = CFG["data"]["base_url"]
OUT = P["raw"]


def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def fetch(url: str, dest: Path, retries: int = 8) -> None:
    dest.parent.mkdir(parents=True, exist_ok=True)
    tmp = dest.with_suffix(dest.suffix + ".part")
    for attempt in range(retries):
        try:
            with urllib.request.urlopen(url, timeout=60) as r, open(tmp, "wb") as f:
                while chunk := r.read(1 << 16):
                    f.write(chunk)
            tmp.replace(dest)
            return
        except Exception as e:
            if attempt == retries - 1:
                raise RuntimeError(f"{url}: {e}") from e
            time.sleep(2 * (attempt + 1))


def mirror(workers: int = 24):
    fetch(BASE_URL + "SHA256SUMS.txt", OUT / "SHA256SUMS.txt")
    entries = [ln.split(maxsplit=1) for ln in (OUT / "SHA256SUMS.txt").read_text().splitlines()]
    entries = [(d, n.strip()) for d, n in entries]
    todo = [(d, n) for d, n in entries if not ((OUT / n).exists() and sha256(OUT / n) == d)]
    print(f"{len(entries)} files listed; {len(entries) - len(todo)} already verified; {len(todo)} to download")

    def job(item):
        d, n = item
        fetch(BASE_URL + n, OUT / n)
        return n, sha256(OUT / n) == d

    bad = []
    with ThreadPoolExecutor(workers) as ex:
        for k, fut in enumerate(as_completed([ex.submit(job, it) for it in todo]), 1):
            try:
                n, ok = fut.result()
            except Exception as e:
                n, ok = str(e), False
            if not ok:
                bad.append(n)
            if k % 1000 == 0:
                print(f"  {k}/{len(todo)}")
    return entries, bad


t0 = time.time()
entries, bad = mirror()
print(f"done in {time.time() - t0:.0f}s; failed/mismatched: {len(bad)}")
assert not bad, bad[:10]

10434 files listed; 10434 already verified; 0 to download
done in 121s; failed/mismatched: 0


In [4]:
print("Dataset folder:")
print(OUT)

print("\nExists:", OUT.exists())

if OUT.exists():
    files = list(OUT.rglob("*"))
    print("Files currently there:", sum(f.is_file() for f in files))

Dataset folder:
C:\Users\roshh\Desktop\rl\version 2\data\raw\circor_v1.0.3

Exists: True


Files currently there: 10435


## Integrity report

In [5]:
files = [OUT / n for _, n in entries]
ext = pd.Series([f.suffix for f in files]).value_counts()
size_mb = sum(f.stat().st_size for f in files) / 2**20
report = {
    "version": CFG["data"]["dataset_version"],
    "files_listed_in_SHA256SUMS": len(entries),
    "files_verified": len(entries) - len(bad),
    "total_size_MB": round(size_mb, 1),
    "by_extension": ext.to_dict(),
    "records_in_RECORDS": len((OUT / "RECORDS").read_text().split()),
    "download_command": "wget -r -N -c -np " + BASE_URL,
}
save_json(report, P["results"] / "download_report.json")
print(json.dumps(report, indent=2))
print((OUT / "LICENSE.txt").read_text()[:400])

{
  "version": "1.0.3",
  "files_listed_in_SHA256SUMS": 10434,
  "files_verified": 10434,
  "total_size_MB": 557.9,
  "by_extension": {
    ".hea": 3163,
    ".tsv": 3163,
    ".wav": 3163,
    ".txt": 943,
    "": 1,
    ".csv": 1
  },
  "records_in_RECORDS": 3163,
  "download_command": "wget -r -N -c -np https://physionet.org/files/circor-heart-sound/1.0.3/"
}
# ODC Attribution License (ODC-By)

### Preamble

The Open Data Commons Attribution License is a license agreement
intended to allow users to freely share, modify, and use this Database
subject only to the attribution requirements set out in Section 4.

Databases can contain a wide variety of types of content (images,
audiovisual material, and sounds all in the same database, for example),
and so 
